# Cyclic peptide binder tutorial: from a target to designed macrocycles

In this tutorial you will use RFD4-Proteina to design cyclic peptide binders against a target, measure
whether the ring actually closed, and view the resulting macrocycle–target complexes.

A macrocyclic peptide is a short chain joined head to tail or through a side-chain crosslink. The ring is what makes
it rigid, and the model only knows about it because the input declares the closing bond as a `_struct_conn` record.
The inputs in `docs/examples/inputs/cyclic_binders` pair a target, PD-1, with a 12-residue peptide whose structure and
sequence are both designed, and differ only in how residue 1 is joined to residue 12:

| Chemistry | Closing bond | Ideal distance |
|---|---|---|
| Head-to-tail | C-terminal `C` to N-terminal `N` (an amide) | 1.33 Å |
| Disulfide | `CYS.SG` to `CYS.SG` | 2.05 Å |
| Isopeptide | `LYS.NZ` to `ASP.CG` (an amide) | 1.33 Å |

**The design regime decides the outcome here, more than the chemistry does.** When the ring already has coordinates,
all three chemistries close reliably once the crosslinked residues are atomized — that case is the
[cyclic peptide tutorial](cyclic_peptide_tutorial.ipynb). A *designed* ring has no input coordinates, and an atomized
token must carry a finite coordinate at inference, so that fix is unavailable. Step 3 measures what it costs.

You will:

1. inspect one input and read back what it fixes and what it leaves to the model;
2. generate designs for each of the three chemistries;
3. measure ring closure and hotspot engagement, keeping failures in the denominator;
4. view the designs with the closing bond drawn explicitly.

## Running on Colab

Everything in this tutorial works on Colab: it samples without a refolding oracle. The next cell does the Colab setup
(install the published package for the runtime's CUDA version and download the three inputs this notebook reads; the
model comes from Hugging Face on first use) and is skipped elsewhere. Pick a GPU runtime first:
**Runtime → Change runtime type → L4, A100 or H100**.

In [ ]:
"""On Colab only: install the published package and fetch the few repository files this notebook reads."""
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    import re
    import subprocess
    import urllib.request
    from pathlib import Path

    if sys.version_info[:2] != (3, 12):
        raise SystemExit(f"RFD4-Proteina needs Python 3.12; this Colab runtime has {sys.version.split()[0]}. "
                         "Use the Brev launchable or a local install: https://rfd4-proteina.org/latest/installation.html")
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if smi.returncode != 0:
        raise SystemExit("No GPU attached. Runtime > Change runtime type > L4, A100 or H100 GPU, then run this cell again.")
    cuda = re.search(r"CUDA Version:\s*(\d+)", smi.stdout)
    if not cuda or int(cuda.group(1)) < 12:
        raise SystemExit("Could not find a CUDA 12 or 13 driver in nvidia-smi.")
    extra = "all-cuda13" if int(cuda.group(1)) >= 13 else "all-cuda12"
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.splitlines()[0]
    print("GPU:", gpu, "->", f"rfd4-proteina[{extra}]")
    if float(gpu.rsplit(",", 1)[1]) < 8.0:
        print("WARNING: this GPU predates Ampere (a T4 is Turing) and is not on the qualified-hardware list.")

    # No clone: the published package carries the code and configs; the model bundle comes from Hugging Face on first use.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "--torch-backend=auto", f"rfd4-proteina[{extra}]", "py3Dmol"], check=True)

    # The notebook reads three inputs by repository-relative path: download just those, with the `.project-root` marker.
    RAW = "https://raw.githubusercontent.com/RosettaCommons/RFD4-Proteina/main/"
    work = Path("/content/rfd4-tutorial")
    work.mkdir(parents=True, exist_ok=True)
    (work / ".project-root").touch()
    for rel in ["docs/examples/inputs/cyclic_binders/01_PD1_head_tail_12.cif.zst",
                "docs/examples/inputs/cyclic_binders/01_PD1_disulfide_12.cif.zst",
                "docs/examples/inputs/cyclic_binders/01_PD1_isopeptide_12.cif.zst"]:
        dest = work / rel
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(RAW + rel, dest)
    os.chdir(work)
    os.environ.setdefault("CLUSTER", "local")
    print("Ready in", work, "(if Colab asks to restart the session, restart and re-run this cell)")
else:
    print("Not on Colab: nothing to install. Run this notebook inside a checkout with the environment set up.")

## Setup

The next cell finds the repository, sets the paths and defines the small helpers used below: a function that runs
`rfd4-generate`, a loader, the closure geometry for each chemistry, and viewer helpers. The code is folded by
default.

In [ ]:
import io, os, sys, glob, shutil, subprocess, warnings, logging
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("atomworks").setLevel(logging.ERROR)

# Locate the repository by walking up to the `.project-root` marker; $PROJECT_ROOT is the fallback.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".project-root").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / ".project-root").exists():
    if not os.environ.get("PROJECT_ROOT"):
        raise RuntimeError("Run this notebook inside the RFD4-Proteina checkout, or set PROJECT_ROOT.")
    REPO_ROOT = Path(os.environ["PROJECT_ROOT"])
os.environ["PROJECT_ROOT"] = str(REPO_ROOT)
os.environ.setdefault("CLUSTER", "local")
os.environ.setdefault("ALLOW_BIOTITE_CCD", "True")
INPUT_DIR = REPO_ROOT / "docs" / "examples" / "inputs" / "cyclic_binders"
OUT_ROOT = Path(os.environ.get("CYCLIC_BINDER_TUTORIAL_RUNS", REPO_ROOT / "runs" / "cyclic_binder_tutorial"))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Extra Hydra overrides applied to every run, for example "hf_ckpt_path=null" to use a native checkpoint.
EXTRA_OVERRIDES = os.environ.get("CYCLIC_BINDER_TUTORIAL_OVERRIDES", "").split()

# Subprocesses do not inherit sys.path: let them import the checkout's sources.
os.environ["PYTHONPATH"] = f"{REPO_ROOT / 'src'}:" + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: generation needs a GPU")

PEPTIDE_CHAIN, TARGET_CHAIN, LENGTH = "P", "B", 12
HOTSPOTS = [33, 95, 97, 102]            # PD-1 residues annotated as hotspots in the PPI benchmark

# chemistry -> (atom on residue 1, atom on residue LENGTH, ideal distance, tolerance)
CLOSURE = {
    "head_tail":  ("N",  "C",  1.33, 0.30),
    "disulfide":  ("SG", "SG", 2.05, 0.30),
    "isopeptide": ("NZ", "CG", 1.33, 0.30),
}

# Run the checkout's source when there is one, otherwise the installed `rfd4-generate` (Colab has no checkout).
_SRC_ENTRY = REPO_ROOT / "src/rfproteina/generate.py"
ENTRY = [sys.executable, str(_SRC_ENTRY)] if _SRC_ENTRY.exists() else [shutil.which("rfd4-generate")]


def generate(chemistry, out_dir, *overrides):
    """Run generation for one chemistry and return the generated CIFs."""
    cmd = [*ENTRY,
           f"inputs={INPUT_DIR / f'01_PD1_{chemistry}_{LENGTH:02d}.cif.zst'}",
           f"out_dir={out_dir}", "~metrics", "compile_lengths=null",
           "recycle_strategy=medium/front_loaded", "compress_outputs=false",
           *overrides, *EXTRA_OVERRIDES]
    proc = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-1500:], proc.stderr[-3000:])
        raise RuntimeError("generation failed")
    return sorted(Path(out_dir).glob("inference_samples/*/*_generated.cif"))


from atomworks.io import parse
from rfproteina.utils.parse_presets import ANNOTATED_CIF
import rfproteina.conditions  # registers the condition names used in selections, such as C_HOT


def load(path):
    """Load an annotated CIF (also .cif.zst) as the first model of its first assembly."""
    return parse(str(path), config=ANNOTATED_CIF)["assemblies"]["1"][0]


def closure_distance(aa, chemistry):
    """Distance between the two atoms the input declared as bonded, or nan if either is absent."""
    atom_a, atom_b, _, _ = CLOSURE[chemistry]
    pep = aa.chain_id == PEPTIDE_CHAIN

    def coord(res_id, name):
        sel = pep & (aa.res_id == res_id) & (aa.atom_name == name)
        return aa.coord[sel][0] if sel.sum() == 1 else None

    a, b = coord(1, atom_a), coord(LENGTH, atom_b)
    return float(np.linalg.norm(a - b)) if a is not None and b is not None else float("nan")


def hotspots_engaged(aa, cutoff=5.0):
    """How many hotspot residues the peptide reaches, and how many atom contacts it makes."""
    pep, tgt = aa.chain_id == PEPTIDE_CHAIN, aa.chain_id == TARGET_CHAIN
    hot = tgt & np.isin(aa.res_id, HOTSPOTS)
    d_hot = np.linalg.norm(aa.coord[pep][:, None, :] - aa.coord[hot][None, :, :], axis=-1)
    # Count hotspot *residues* in contact, not their atoms: one residue contributes many atoms.
    residues = {int(r) for r, dd in zip(aa.res_id[hot], d_hot.min(axis=0)) if dd <= cutoff}
    d_all = np.linalg.norm(aa.coord[pep][:, None, :] - aa.coord[tgt][None, :, :], axis=-1)
    return len(residues), int((d_all <= cutoff).sum())


import py3Dmol
import biotite.structure.io.pdb as biopdb

TARGET, PEPTIDE, HOTSPOT, BOND = "0xBBBBBB", "0x2A9D8F", "0x7B2D8E", "0xE76F51"


def to_pdb(atoms):
    atoms = atoms.copy()
    atoms.set_annotation("b_factor", np.zeros(atoms.array_length()))   # model B-factors can overflow the PDB columns
    atoms.set_annotation("occupancy", np.ones(atoms.array_length()))
    f = biopdb.PDBFile(); f.set_structure(atoms)
    buf = io.StringIO(); f.write(buf); return buf.getvalue()

<a id="step-1"></a>

## 1. Inspect the input

The tutorial ships three inputs, one per chemistry, named `01_PD1_<chemistry>_12`. Each is the same two-chain
problem: chain `B` is the PD-1 target and chain `P` the macrocycle. Pick one:

In [ ]:
CHEMISTRY = "head_tail"        # "head_tail", "disulfide" or "isopeptide"

aa_in = load(INPUT_DIR / f"01_PD1_{CHEMISTRY}_{LENGTH:02d}.cif.zst")
print(sorted(p.name for p in INPUT_DIR.glob("*.cif.zst")))

An annotated CIF carries the conditions as per-atom and per-residue annotations (see
[Preparing inputs](../how-to/input_prep/index.md)). Read back what this input fixes: the target keeps its coordinates
and its sequence, while the peptide keeps neither — both are designed.

In [ ]:
rows = []
for chain in np.unique(aa_in.chain_id):
    m = aa_in.chain_id == chain
    rows.append({
        "chain": chain,
        "residues": len(set(aa_in.res_id[m])),
        "atoms": int(m.sum()),
        "coordinates fixed (C_CRD)": int((m & aa_in.mask("sa_mask('C_CRD')")).sum()),
        "sequence fixed (C_SEQ)": int((m & aa_in.mask("sa_mask('C_SEQ')")).sum()),
        "hotspot atoms (C_HOT)": int((m & aa_in.mask("sa_mask('C_HOT')")).sum()),
    })
pd.DataFrame(rows).set_index("chain")

The closing bond is the one thing that makes this a macrocycle rather than a linear peptide, and it
reaches the model through the bond graph. A linear 12-residue peptide would have one bond fewer than it has atoms;
the extra bond here is the closure:

In [ ]:
pep = aa_in.chain_id == PEPTIDE_CHAIN
atom_a, atom_b, ideal, _ = CLOSURE[CHEMISTRY]
print(f"peptide atoms {int(pep.sum())}, bonds in the complex {len(aa_in.bonds.as_array())}")
print(f"declared closure: {atom_a}1 - {atom_b}{LENGTH}   (ideal {ideal} A)")

Three ways to author an input like this wrongly. Each produces **no error**: the run exits `0` and
writes a plausible file.

- **`expseg` bounds default to `-1`.** `-1` marks a chain that is not an expandable segment; `0` means "expand to
  zero residues", and the chain is realized away before generation — leaving a complex with no peptide in it.
- **Conditions must be registered before parsing or writing.** Without `import rfproteina.conditions` the condition
  classes are unknown and the hotspots are dropped on write.
- **Declare closures as `covale`.** The parse preset reads `covale` and `metalc`; an RCSB `disulf` record is ignored,
  and a length-range binder (`80-150`, one `UNK` atom) has no numbered residues for `_struct_conn` to name at all,
  which is why the peptide chain here is enumerated explicitly.

Validate before spending GPU time, and look for `struct_conn_ok ... all covale linkages verified in the bond graph`:

```bash
bash skills/rfd4-proteina-annotated-cif/scripts/check_annotated_cif.sh <input.cif>
```

The peptide has no coordinates yet, so there is nothing of it to draw. This is the target it has to
bind, with the hotspots the model is asked to contact:

In [ ]:
v = py3Dmol.view(width=720, height=420)
v.addModel(to_pdb(aa_in[(aa_in.chain_id == TARGET_CHAIN) & ~aa_in.mask("has_nan_coord()")]), "pdb")
v.setStyle({"cartoon": {"color": TARGET}})
v.addStyle({"resi": HOTSPOTS}, {"stick": {"color": HOTSPOT, "radius": 0.3}})
v.setBackgroundColor("0xffffff"); v.zoomTo()
print("gray: PD-1 target   purple: hotspots (the macrocycle is still to be designed)")
v.show()

<a id="step-2"></a>

## 2. Generate designs

PPI work uses a front-loaded recycling schedule; `medium/front_loaded` is the medium preset from the
[recycling schedule table](../how-to/run-the-model.md#recycling-schedules). We draw a few candidates for each
chemistry.

| Setting | Config key | Effect |
|---|---|---|
| `NUM_REPLICATES` | `num_replicates` | Candidates per input. A few is a pilot; 100 or more is a production run. |
| `BATCH_SIZE` | `batch_size` | Candidates sampled together on the GPU. |
| `SEED` | `seed` | Makes the run repeatable. |

In [ ]:
NUM_REPLICATES = int(os.environ.get("CYCLIC_BINDER_TUTORIAL_REPLICATES", 4))
BATCH_SIZE = 1
SEED = 0

designs = {}
for chemistry in CLOSURE:
    out_dir = OUT_ROOT / chemistry
    found = sorted(out_dir.glob("inference_samples/*/*_generated.cif"))
    designs[chemistry] = found or generate(
        chemistry, out_dir,
        f"num_replicates={NUM_REPLICATES}", f"batch_size={BATCH_SIZE}", f"seed={SEED}",
    )
    print(f"{chemistry:<11} {len(designs[chemistry])} designs -> {out_dir}")

<a id="step-3"></a>

## 3. Measure closure and binding

Two separate questions, and conflating them is easy: a ring that closes in empty space is not a binder, and a peptide
packed against the hotspots with an open ring is not a macrocycle.

The closure test is **two-sided**, `|d − ideal| ≤ 0.3 Å`. A one-sided "closed if under 2.3 Å" cut would score an
impossible overshoot as a success — an un-atomized disulfide can land near 1.2 Å, well inside a single sulfur atom.
Designs whose closure atoms are missing are unmeasurable and stay in the denominator.

In [ ]:
rows = []
for chemistry, paths in designs.items():
    _, _, ideal, tol = CLOSURE[chemistry]
    loaded = [load(p) for p in paths]
    d = np.array([closure_distance(aa, chemistry) for aa in loaded])
    engaged = [hotspots_engaged(aa) for aa in loaded]
    finite = d[np.isfinite(d)]
    rows.append({
        "chemistry": chemistry,
        "closed": f"{int((np.abs(finite - ideal) <= tol).sum())}/{len(d)}",
        "median (A)": round(float(np.median(finite)), 3) if len(finite) else None,
        "range (A)": f"{finite.min():.2f}-{finite.max():.2f}" if len(finite) else None,
        "ideal (A)": ideal,
        "hotspots engaged": f"{int(np.median([e[0] for e in engaged]))} of {len(HOTSPOTS)}",
        "contacts < 5 A": int(np.median([e[1] for e in engaged])),
    })
pd.DataFrame(rows).set_index("chemistry")

<a id="step-4"></a>

## 4. View the designs

Target in gray, hotspots in purple, the macrocycle in teal, and the declared closing bond drawn as an orange dashed
line between the two atoms it names. Where that line is long, the ring did not close.

In [ ]:
def show(chemistry, index=0, width=720, height=440):
    aa = load(designs[chemistry][index])
    atom_a, atom_b, ideal, tol = CLOSURE[chemistry]
    d = closure_distance(aa, chemistry)
    pep = aa.chain_id == PEPTIDE_CHAIN

    v = py3Dmol.view(width=width, height=height)
    v.addModel(to_pdb(aa[aa.chain_id == TARGET_CHAIN]), "pdb")
    v.setStyle({"cartoon": {"color": TARGET}})
    v.addStyle({"resi": HOTSPOTS}, {"stick": {"color": HOTSPOT, "radius": 0.3}})

    v.addModel(to_pdb(aa[pep]), "pdb")
    v.setStyle({"model": -1}, {"cartoon": {"color": PEPTIDE}, "stick": {"color": PEPTIDE, "radius": 0.15}})

    ends = [aa.coord[pep & (aa.res_id == r) & (aa.atom_name == n)]
            for r, n in ((1, atom_a), (LENGTH, atom_b))]
    if all(len(e) for e in ends):
        v.addLine({"start": dict(zip("xyz", map(float, ends[0][0]))),
                   "end": dict(zip("xyz", map(float, ends[1][0]))),
                   "color": BOND, "dashed": True, "linewidth": 4})
    v.setBackgroundColor("0xffffff"); v.zoomTo({"model": -1})
    verdict = "closed" if abs(d - ideal) <= tol else "open"
    print(f"{chemistry}: {atom_a}1-{atom_b}{LENGTH} = {d:.3f} A, ideal {ideal} -> {verdict}")
    v.show()


show("head_tail")

In [ ]:
show("disulfide")

In [ ]:
show("isopeptide")

<a id="next-steps"></a>

## Next steps

You designed cyclic peptide binders for one target, measured ring closure against hotspot engagement, and saw where
each chemistry stands. Measured over 16 replicates on the released checkpoint, the three behave very differently:

| Chemistry | Closed | Median (Å) | Range (Å) | Hotspots engaged |
|---|---|---|---|---|
| head-to-tail | **16/16** | 1.318 | 1.277–1.390 | 4 of 4 |
| disulfide | 12/16 | 2.119 | 1.576–2.724 | 4 of 4 |
| isopeptide | **1/16** | 2.136 | 1.478–4.412 | 4 of 4 |

All three dock: every chemistry engages all four hotspots with a comparable interface, so closure is what separates
them, not binding.

1. **Design with head-to-tail.** It closed 16/16 here at 1.318 Å against an ideal of 1.33, with a 0.11 Å spread, and
   over ring lengths 6–20 it closes 360/360 at a median of 1.308 Å with no length dependence.
2. **Treat the disulfide as a filter, not a guarantee.** It closed 12/16 here and 177/360 over lengths 6–20, best
   between 10 and 16 residues and failing at 6–7 where the ring cannot take the bridge. Its median sits on the ideal,
   so the problem is spread rather than aim: sample more candidates and select on the measured distance.
3. **Do not expect the isopeptide to close without input coordinates.** With coordinates on the ring it is reliable
   (24/24 at 1.363 Å once the crosslinked residues are atomized), so this is the regime rather than the chemistry.
   Over lengths 6–20 the model usually closed onto the *wrong atom*, placing `LYS.NZ` against the carboxyl oxygen
   `OD1` (median 1.614 Å, the closest contact in 243 of 360 designs) instead of forming the `NZ`–`CG` amide
   (2.325 Å) — which a threshold on the declared pair alone cannot distinguish from a ring that never closed.
4. **Score the survivors.** Ring closure is geometry, not designability: nothing here was refolded. Run the
   `interface` metrics with an AlphaFold 3 or OpenFold3 refold before believing any of these are binders. See
   [Evaluate with metrics](../how-to/evaluate-with-metrics.md) and [Refolding](../how-to/refolding.md).

To retarget, build the same two-chain input against another PPI benchmark target: keep the target's coordinate,
sequence and hotspot conditioning, enumerate the peptide chain explicitly, and declare its closing bond as `covale`.